In [35]:
import sys
from pathlib import Path
import numpy as np

root = Path.cwd()
candidates = [root, root.parent, root.parent.parent]
for base in candidates:
    if (base / "efgp_eigenpro_py").exists():
        sys.path.insert(0, str(base))
        break
else:
    raise RuntimeError("cannot find efgp_eigenpro_py from current path")

from efgp_eigenpro_py.efgp_solver import EFGPSolver
from efgp_eigenpro_py.kernels import make_squared_exponential
from efgp_eigenpro_py.eigenspace import estimate_top_eigenspace
from efgp_eigenpro_py.eigenpro_precond import build_preconditioner

import time

np.set_printoptions(precision=3, suppress=True)


In [36]:
def make_data_1d(n=30, noise=0.0, seed=0):
    rng = np.random.default_rng(seed)
    x = np.sort(rng.uniform(0.0, 1.0, size=n))[:, None]
    f = np.sin(2 * np.pi * x[:, 0]) + 0.3 * np.cos(6 * np.pi * x[:, 0])
    y = f + noise * rng.standard_normal(n)
    return x, y, f


def make_data_2d(n=40, noise=0.0, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0.0, 1.0, size=(n, 2))
    f = (
        np.sin(2 * np.pi * x[:, 0])
        + 0.5 * np.cos(2 * np.pi * x[:, 1])
        + 0.2 * np.sin(2 * np.pi * (x[:, 0] + x[:, 1]))
    )
    y = f + noise * rng.standard_normal(n)
    return x, y, f


In [37]:
def dense_multi_index(m, dim):
    grid = [np.arange(-m, m + 1)] * dim
    mesh = np.meshgrid(*grid, indexing="ij")
    return np.stack([g.reshape(-1) for g in mesh], axis=1)


def build_dense_phi(solver, x, state):
    x = solver._ensure_2d(x)
    tphx = 2.0 * np.pi * state.grid.h * (x - state.x_center)

    m = (state.grid.mtot - 1) // 2
    J = dense_multi_index(m, solver.kernel.dim)
    w = state.weights.reshape(-1)

    phase = np.exp(1j * (tphx @ J.T))
    Phi = phase * w[None, :]
    return Phi


def build_dense_A_rhs(solver, x, y, state):
    Phi = build_dense_phi(solver, x, state)
    A = Phi.conj().T @ Phi + solver.reg_lambda * np.eye(Phi.shape[1], dtype=complex)
    rhs = Phi.conj().T @ y
    return Phi, A, rhs


In [38]:
def build_precond_from_state(solver, state, top_q):
    size = state.rhs.size
    if top_q is None or top_q <= 0:
        return None, None, None, (0.0, 0.0), top_q
    if top_q >= size:
        top_q = max(1, size - 1)

    t0 = time.perf_counter()
    eigpairs = estimate_top_eigenspace(
        lambda v: solver._apply_A(state, v),
        size=size,
        top_q=top_q,
    )
    t1 = time.perf_counter()

    if eigpairs.values.size > top_q:
        mu = eigpairs.values[top_q]
    else:
        mu = eigpairs.values[-1]

    precond = build_preconditioner(
        eigpairs.values[:top_q],
        eigpairs.vectors[:, :top_q],
        mu,
    ).apply
    t2 = time.perf_counter()

    return precond, eigpairs, mu, (t1 - t0, t2 - t1), top_q


def richardson_history(matvec, b, x0, eta, maxiter, precond=None):
    x = x0.copy()
    r = matvec(x) - b
    hist = [np.linalg.norm(r) / max(np.linalg.norm(b), 1e-15)]
    for _ in range(maxiter):
        z = precond(r) if precond is not None else r
        Az = matvec(z)
        x = x - eta * z
        r = r - eta * Az
        hist.append(np.linalg.norm(r) / max(np.linalg.norm(b), 1e-15))
    return x, np.array(hist)


def pcg_history(matvec, b, tol, maxiter, precond=None):
    try:
        from scipy.sparse.linalg import LinearOperator, cg  # noqa: F401
    except Exception as exc:
        print("scipy not usable, skip pcg:", exc)
        return None, None, None, 0

    n = b.size
    dtype = np.result_type(b.dtype, np.complex128)
    Aop = LinearOperator((n, n), matvec=matvec, dtype=dtype)
    Mop = None if precond is None else LinearOperator((n, n), matvec=precond, dtype=dtype)

    hist = []

    def cb(xk):
        rk = matvec(xk) - b
        hist.append(np.linalg.norm(rk) / max(np.linalg.norm(b), 1e-15))

    x, info = cg(Aop, b, rtol=tol, atol=0.0, maxiter=maxiter, M=Mop, callback=cb)
    if len(hist) == 0:
        rk = matvec(x) - b
        hist.append(np.linalg.norm(rk) / max(np.linalg.norm(b), 1e-15))
    return x, np.array(hist), info, len(hist)


def print_history_summary(name, hist, info=None):
    if hist is None:
        print(name, "skipped")
        return
    final_relres = float(hist[-1])
    n_iter = int(len(hist) - 1)
    if info is None:
        print(f"{name}: it={n_iter} final_relres={final_relres:.3e}")
    else:
        print(f"{name}: it={n_iter} final_relres={final_relres:.3e} info={info}")


def spectral_metrics(A_dense, eigpairs=None, mu=None, top_q=None):
    evals = np.linalg.eigvalsh(A_dense)
    lam_min = float(evals[0])
    lam_max = float(evals[-1])
    cond_A = lam_max / lam_min

    evals_desc = evals[::-1]
    lam1 = float(evals_desc[0])
    lam_q = float(evals_desc[top_q - 1]) if top_q is not None and top_q <= evals_desc.size else None
    lam_q1 = float(evals_desc[top_q]) if top_q is not None and top_q < evals_desc.size else None

    print("lambda_max =", lam1)
    print("lambda_min =", lam_min)
    if lam_q is not None:
        print("lambda_q =", lam_q)
    if lam_q1 is not None:
        print("lambda_q_plus_1 =", lam_q1)
    print("cond(A) =", cond_A)

    if eigpairs is None or mu is None or top_q is None:
        return

    Uq = eigpairs.vectors[:, :top_q]
    theta = eigpairs.values[:top_q]
    scale = 1.0 - (mu / theta)
    P_dense = np.eye(A_dense.shape[0], dtype=complex) - Uq @ np.diag(scale) @ Uq.conj().T
    PA = P_dense @ A_dense
    PA_sym = 0.5 * (PA + PA.conj().T)
    evals_pa = np.linalg.eigvalsh(PA_sym)
    cond_pa = float(evals_pa[-1] / evals_pa[0])
    print("cond(PA) =", cond_pa)

    head = min(15, evals_desc.size)
    print("top eigvals(A) =", evals_desc[:head])
    head_pa = min(15, evals_pa.size)
    print("top eigvals(PA) =", evals_pa[::-1][:head_pa])


def make_data_3d(n=20, noise=0.0, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0.0, 1.0, size=(n, 3))
    f = true_func_3d(x)
    y = f + noise * rng.standard_normal(n)
    return x, y, f


def true_func_1d(x):
    x = np.asarray(x).reshape(-1)
    return np.sin(2 * np.pi * x) + 0.3 * np.cos(6 * np.pi * x)


def true_func_2d(x):
    x = np.asarray(x)
    return (
        np.sin(2 * np.pi * x[:, 0])
        + 0.5 * np.cos(2 * np.pi * x[:, 1])
        + 0.2 * np.sin(2 * np.pi * (x[:, 0] + x[:, 1]))
    )


def true_func_3d(x):
    x = np.asarray(x)
    return (
        np.sin(2 * np.pi * x[:, 0])
        + 0.5 * np.cos(2 * np.pi * x[:, 1])
        + 0.2 * np.sin(2 * np.pi * x[:, 2])
        + 0.1 * np.cos(2 * np.pi * (x[:, 0] + x[:, 1]))
    )


def compute_rmse(yhat, ytrue):
    yhat = np.asarray(yhat)
    ytrue = np.asarray(ytrue)
    return float(np.sqrt(np.mean((yhat - ytrue) ** 2)))


def train_test_rmse(solver, x_train, y_train, x_test, y_test, top_q=None, use_richardson=False, eta=None):
    beta, state = solver.solve(
        x_train,
        y_train,
        top_q=top_q,
        use_richardson=use_richardson,
        eta=eta if eta is not None else 0.8,
        cg_tol=1e-10,
        cg_maxiter=5000,
    )
    yhat_train = solver.predict(x_train, beta, state)
    yhat_test = solver.predict(x_test, beta, state)
    rmse_train = compute_rmse(yhat_train, y_train)
    rmse_test = compute_rmse(yhat_test, y_test)
    return rmse_train, rmse_test


def eigenspace_estimation_metrics(solver, x, y, top_q=5):
    state = solver.precompute(x, y)
    _, A_dense, _ = build_dense_A_rhs(solver, x, y, state)

    evals, evecs = np.linalg.eigh(A_dense)
    evals_desc = evals[::-1]
    evecs_desc = evecs[:, ::-1]

    eigpairs = estimate_top_eigenspace(
        lambda v: solver._apply_A(state, v),
        size=state.rhs.size,
        top_q=top_q,
    )

    true_vals = evals_desc[: top_q + 1]
    est_vals = eigpairs.values[: top_q + 1]
    rel_err = np.abs(est_vals - true_vals) / np.maximum(np.abs(true_vals), 1e-15)

    U_true = evecs_desc[:, :top_q]
    U_est = eigpairs.vectors[:, :top_q]
    proj_true = U_true @ U_true.conj().T
    proj_est = U_est @ U_est.conj().T
    proj_err_fro = np.linalg.norm(proj_est - proj_true, ord="fro")

    s = np.linalg.svd(U_true.conj().T @ U_est, compute_uv=False)
    sin_theta = float(np.sqrt(max(0.0, 1.0 - float(np.min(s)) ** 2)))

    print("eigval relerr (top q+1) =", rel_err)
    print("proj_err_fro =", proj_err_fro)
    print("sin_theta_max =", sin_theta)

    return {
        "eigval_relerr": rel_err,
        "proj_err_fro": proj_err_fro,
        "sin_theta_max": sin_theta,
    }


def richardson_eta_sweep(solver, x, y, top_q=5, c_list=None, maxiter=200):
    if c_list is None:
        c_list = [0.2, 0.5, 0.8, 1.0, 1.2]

    state = solver.precompute(x, y)
    _, A_dense, rhs_dense = build_dense_A_rhs(solver, x, y, state)
    matvec = lambda v: solver._apply_A(state, v)

    precond, _, mu, _, top_q_used = build_precond_from_state(solver, state, top_q)
    if mu is None:
        print("mu not available, skip eta sweep")
        return

    beta_ref = np.linalg.solve(A_dense, rhs_dense)
    x0 = np.zeros_like(rhs_dense)

    print("eta sweep using top_q =", top_q_used, "mu =", float(mu))
    for c in c_list:
        eta = float(c) / float(mu)
        beta, hist = richardson_history(matvec, rhs_dense, x0, eta, maxiter=maxiter, precond=precond)
        relres = float(hist[-1])
        beta_err = np.linalg.norm(beta - beta_ref) / np.linalg.norm(beta_ref)
        print("c=", c, "eta=", eta, "final_relres=", relres, "beta_err=", beta_err)


def print_metrics_table(rows, title):
    if not rows:
        return
    keys = list(rows[0].keys())
    widths = {k: len(k) for k in keys}
    for row in rows:
        for k in keys:
            widths[k] = max(widths[k], len(str(row.get(k, ""))))

    print(title)
    header = " | ".join(k.ljust(widths[k]) for k in keys)
    sep = "-+-".join("-" * widths[k] for k in keys)
    print(header)
    print(sep)
    for row in rows:
        line = " | ".join(str(row.get(k, "")).ljust(widths[k]) for k in keys)
        print(line)


def sweep_params_2d():
    x, y, _ = make_data_2d(n=30, noise=0.0, seed=11)
    top_q_list = [2, 5, 10, 20]
    reg_list = [1e-2, 1e-3, 1e-4]
    eps_list = [2e-3, 1e-3]
    l_list = [0.2, 0.4]

    for lengthscale in l_list:
        for reg_lambda in reg_list:
            for eps in eps_list:
                print("lengthscale=", lengthscale, "reg_lambda=", reg_lambda, "eps=", eps)
                for top_q in top_q_list:
                    solver = EFGPSolver(
                        make_squared_exponential(lengthscale=lengthscale, dim=2, variance=1.0),
                        reg_lambda=reg_lambda,
                        eps=eps,
                        nufft_tol=1e-12,
                    )
                    state = solver.precompute(x, y)
                    _, A_dense, rhs_dense = build_dense_A_rhs(solver, x, y, state)
                    matvec = lambda v: solver._apply_A(state, v)

                    precond, _, _, _, top_q_used = build_precond_from_state(solver, state, top_q)

                    xcg, hcg, info_cg, _ = pcg_history(matvec, rhs_dense, tol=1e-10, maxiter=200, precond=None)
                    xcgp, hcgp, info_pcg, _ = pcg_history(matvec, rhs_dense, tol=1e-10, maxiter=200, precond=precond)

                    beta_ref = np.linalg.solve(A_dense, rhs_dense)
                    beta_err_plain = None
                    beta_err_prec = None
                    if xcg is not None:
                        beta_err_plain = np.linalg.norm(xcg - beta_ref) / np.linalg.norm(beta_ref)
                    if xcgp is not None:
                        beta_err_prec = np.linalg.norm(xcgp - beta_ref) / np.linalg.norm(beta_ref)

                    plain_iter = len(hcg) - 1 if hcg is not None else None
                    prec_iter = len(hcgp) - 1 if hcgp is not None else None
                    plain_relres = float(hcg[-1]) if hcg is not None else None
                    prec_relres = float(hcgp[-1]) if hcgp is not None else None

                    print(
                        "top_q=", top_q_used,
                        "plain_it=", plain_iter,
                        "prec_it=", prec_iter,
                        "plain_relres=", plain_relres,
                        "prec_relres=", prec_relres,
                        "plain_beta_err=", beta_err_plain,
                        "prec_beta_err=", beta_err_prec,
                        "info_plain=", info_cg,
                        "info_prec=", info_pcg,
                    )


def check_3d_minimal():
    print("=== 3D minimal sanity check ===")
    x3, y3, _ = make_data_3d(n=18, noise=0.0, seed=2)
    kernel3 = make_squared_exponential(lengthscale=0.4, dim=3, variance=1.0)
    solver3 = EFGPSolver(kernel3, reg_lambda=1e-3, eps=5e-3, nufft_tol=1e-12)
    check_solver_correctness_no_precond(solver3, x3, y3)


In [39]:
def check_solver_correctness_no_precond(solver, x, y):
    state = solver.precompute(x, y)
    Phi, A_dense, rhs_dense = build_dense_A_rhs(solver, x, y, state)

    rhs_err = np.linalg.norm(state.rhs - rhs_dense) / np.linalg.norm(rhs_dense)

    rng = np.random.default_rng(123)
    v = rng.normal(size=rhs_dense.size) + 1j * rng.normal(size=rhs_dense.size)
    Av_fft = solver._apply_A(state, v)
    Av_dense = A_dense @ v
    A_err = np.linalg.norm(Av_fft - Av_dense) / np.linalg.norm(Av_dense)

    beta, state2 = solver.solve(
        x,
        y,
        top_q=None,
        use_richardson=False,
        cg_tol=1e-10,
        cg_maxiter=5000,
    )
    beta_ref = np.linalg.solve(A_dense, rhs_dense)
    beta_err = np.linalg.norm(beta - beta_ref) / np.linalg.norm(beta_ref)

    x_test = x.copy()
    yhat = solver.predict(x_test, beta, state2)
    yhat_ref = np.real(build_dense_phi(solver, x_test, state2) @ beta_ref)
    pred_err = np.linalg.norm(yhat - yhat_ref) / np.linalg.norm(yhat_ref)

    print(f"rhs relerr     = {rhs_err:.3e}")
    print(f"apply_A relerr = {A_err:.3e}")
    print(f"beta relerr    = {beta_err:.3e}")
    print(f"predict relerr = {pred_err:.3e}")

    return {
        "state": state2,
        "Phi": Phi,
        "A_dense": A_dense,
        "rhs_dense": rhs_dense,
        "beta": beta,
        "beta_ref": beta_ref,
        "rhs_err": rhs_err,
        "A_err": A_err,
        "beta_err": beta_err,
        "pred_err": pred_err,
    }


In [40]:
def check_solver_correctness_richardson(solver, x, y):
    state = solver.precompute(x, y)
    _, A_dense, rhs_dense = build_dense_A_rhs(solver, x, y, state)

    eta = 0.9 / np.linalg.norm(A_dense, 2)

    beta, state2 = solver.solve(
        x,
        y,
        top_q=None,
        use_richardson=True,
        eta=eta,
        cg_tol=1e-8,
        cg_maxiter=5000,
    )
    beta_ref = np.linalg.solve(A_dense, rhs_dense)
    beta_err = np.linalg.norm(beta - beta_ref) / np.linalg.norm(beta_ref)

    x_test = x.copy()
    yhat = solver.predict(x_test, beta, state2)
    yhat_ref = np.real(build_dense_phi(solver, x_test, state2) @ beta_ref)
    pred_err = np.linalg.norm(yhat - yhat_ref) / np.linalg.norm(yhat_ref)

    print(f"richardson beta relerr    = {beta_err:.3e}")
    print(f"richardson predict relerr = {pred_err:.3e}")
    return {
        "beta_err": beta_err,
        "pred_err": pred_err,
    }


In [41]:
def check_solver_correctness_with_precond(solver, x, y, top_q=10):
    state = solver.precompute(x, y)
    _, A_dense, rhs_dense = build_dense_A_rhs(solver, x, y, state)

    size = state.rhs.size
    if top_q >= size:
        top_q = max(1, size - 1)
    print("use top_q =", top_q)

    beta_prec, state2 = solver.solve(
        x,
        y,
        top_q=top_q,
        use_richardson=False,
        cg_tol=1e-10,
        cg_maxiter=5000,
    )

    beta_ref = np.linalg.solve(A_dense, rhs_dense)
    beta_err = np.linalg.norm(beta_prec - beta_ref) / np.linalg.norm(beta_ref)

    x_test = x.copy()
    yhat = solver.predict(x_test, beta_prec, state2)
    yhat_ref = np.real(build_dense_phi(solver, x_test, state2) @ beta_ref)
    pred_err = np.linalg.norm(yhat - yhat_ref) / np.linalg.norm(yhat_ref)

    print(f"precond beta relerr    = {beta_err:.3e}")
    print(f"precond predict relerr = {pred_err:.3e}")
    return {
        "beta_err": beta_err,
        "pred_err": pred_err,
        "top_q": top_q,
    }


In [42]:
def check_solver_convergence_metrics(solver, x, y, top_q=5, pcg_tol=1e-10, pcg_maxiter=200, rich_maxiter=200):
    t0 = time.perf_counter()
    state = solver.precompute(x, y)
    t1 = time.perf_counter()

    _, A_dense, rhs_dense = build_dense_A_rhs(solver, x, y, state)
    matvec = lambda v: solver._apply_A(state, v)

    precond, eigpairs, mu, times, top_q_used = build_precond_from_state(solver, state, top_q)
    eig_time, precond_time = times

    print("precompute_time =", t1 - t0)
    print("eigenspace_time =", eig_time)
    print("precond_time    =", precond_time)

    spectral_metrics(A_dense, eigpairs, mu, top_q_used)

    eta_plain = 0.9 / np.linalg.norm(A_dense, 2)
    eta_prec = eta_plain if mu is None else 0.9 / float(mu)

    x0 = np.zeros_like(rhs_dense)
    t_r0 = time.perf_counter()
    xr, hr = richardson_history(matvec, rhs_dense, x0, eta_plain, maxiter=rich_maxiter, precond=None)
    t_r1 = time.perf_counter()
    t_rp0 = time.perf_counter()
    xrp, hrp = richardson_history(matvec, rhs_dense, x0, eta_prec, maxiter=rich_maxiter, precond=precond)
    t_rp1 = time.perf_counter()

    t_cg0 = time.perf_counter()
    xcg, hcg, info_cg, _ = pcg_history(matvec, rhs_dense, tol=pcg_tol, maxiter=pcg_maxiter, precond=None)
    t_cg1 = time.perf_counter()
    t_cgp0 = time.perf_counter()
    xcgp, hcgp, info_pcg, _ = pcg_history(matvec, rhs_dense, tol=pcg_tol, maxiter=pcg_maxiter, precond=precond)
    t_cgp1 = time.perf_counter()

    print_history_summary("Richardson", hr)
    print_history_summary("Precond Richardson", hrp)
    print_history_summary("CG", hcg, info=info_cg)
    print_history_summary("PCG", hcgp, info=info_pcg)

    def fmt_num(v):
        if v is None:
            return "na"
        return f"{v:.3e}"

    def fmt_time(v):
        if v is None:
            return "na"
        return f"{v:.3f}"

    rows = [
        {
            "method": "Richardson",
            "plain_it": len(hr) - 1,
            "prec_it": len(hrp) - 1,
            "plain_relres": fmt_num(float(hr[-1])),
            "prec_relres": fmt_num(float(hrp[-1])),
            "solve_time_plain": fmt_time(t_r1 - t_r0),
            "solve_time_prec": fmt_time(t_rp1 - t_rp0),
        },
        {
            "method": "PCG",
            "plain_it": "na" if hcg is None else (len(hcg) - 1),
            "prec_it": "na" if hcgp is None else (len(hcgp) - 1),
            "plain_relres": "na" if hcg is None else fmt_num(float(hcg[-1])),
            "prec_relres": "na" if hcgp is None else fmt_num(float(hcgp[-1])),
            "solve_time_plain": fmt_time(t_cg1 - t_cg0),
            "solve_time_prec": fmt_time(t_cgp1 - t_cgp0),
        },
    ]
    print_metrics_table(rows, "convergence metrics")

    timing_rows = [
        {"metric": "precompute_time", "value": fmt_time(t1 - t0)},
        {"metric": "eigenspace_time", "value": fmt_time(eig_time)},
        {"metric": "precond_time", "value": fmt_time(precond_time)},
        {"metric": "solve_time_rich_plain", "value": fmt_time(t_r1 - t_r0)},
        {"metric": "solve_time_rich_prec", "value": fmt_time(t_rp1 - t_rp0)},
        {"metric": "solve_time_cg_plain", "value": fmt_time(t_cg1 - t_cg0)},
        {"metric": "solve_time_cg_prec", "value": fmt_time(t_cgp1 - t_cgp0)},
    ]
    print_metrics_table(timing_rows, "timing metrics")

    return {
        "hr": hr,
        "hrp": hrp,
        "hcg": hcg,
        "hcgp": hcgp,
    }


def sweep_params_1d():
    x, y, _ = make_data_1d(n=25, noise=0.0, seed=7)
    top_q_list = [2, 5, 10]
    reg_list = [1e-2, 1e-3]
    eps_list = [1e-3, 1e-4]

    for reg_lambda in reg_list:
        for eps in eps_list:
            print("reg_lambda =", reg_lambda, "eps =", eps)
            for top_q in top_q_list:
                solver = EFGPSolver(
                    make_squared_exponential(lengthscale=0.2, dim=1, variance=1.0),
                    reg_lambda=reg_lambda,
                    eps=eps,
                    nufft_tol=1e-12,
                )
                state = solver.precompute(x, y)
                _, A_dense, rhs_dense = build_dense_A_rhs(solver, x, y, state)
                matvec = lambda v: solver._apply_A(state, v)

                precond, _, _, _, top_q_used = build_precond_from_state(solver, state, top_q)

                xcg, hcg, info_cg, _ = pcg_history(matvec, rhs_dense, tol=1e-10, maxiter=200, precond=None)
                xcgp, hcgp, info_pcg, _ = pcg_history(matvec, rhs_dense, tol=1e-10, maxiter=200, precond=precond)

                beta_ref = np.linalg.solve(A_dense, rhs_dense)

                beta_err_plain = None
                beta_err_prec = None
                if xcg is not None:
                    beta_err_plain = np.linalg.norm(xcg - beta_ref) / np.linalg.norm(beta_ref)
                if xcgp is not None:
                    beta_err_prec = np.linalg.norm(xcgp - beta_ref) / np.linalg.norm(beta_ref)

                plain_iter = len(hcg) - 1 if hcg is not None else None
                prec_iter = len(hcgp) - 1 if hcgp is not None else None
                plain_relres = float(hcg[-1]) if hcg is not None else None
                prec_relres = float(hcgp[-1]) if hcgp is not None else None

                print(
                    "top_q=", top_q_used,
                    "plain_it=", plain_iter,
                    "prec_it=", prec_iter,
                    "plain_relres=", plain_relres,
                    "prec_relres=", prec_relres,
                    "plain_beta_err=", beta_err_plain,
                    "prec_beta_err=", beta_err_prec,
                    "info_plain=", info_cg,
                    "info_prec=", info_pcg,
                )


In [43]:
def build_dense_kernel_matrix(kernel, x):
    x = np.asarray(x)
    diff = x[:, None, :] - x[None, :, :]
    r = np.linalg.norm(diff, axis=-1)
    return kernel.k(r)


def compare_with_original_krr(solver, x, y):
    state = solver.precompute(x, y)
    beta, _ = solver.solve(x, y, top_q=None, use_richardson=False, cg_tol=1e-10, cg_maxiter=5000)
    yhat_efgp = solver.predict(x, beta, state)

    K = build_dense_kernel_matrix(solver.kernel, x)
    alpha = np.linalg.solve(K + solver.reg_lambda * np.eye(K.shape[0]), y)
    yhat_krr = K @ alpha

    rel = np.linalg.norm(yhat_efgp - yhat_krr) / np.linalg.norm(yhat_krr)
    print(f"EFGP vs original KRR train prediction relerr = {rel:.3e}")


In [44]:
print("=== 1D sanity check ===")
x1, y1, _ = make_data_1d(n=30, noise=0.0, seed=0)
kernel1 = make_squared_exponential(lengthscale=0.2, dim=1, variance=1.0)
solver1 = EFGPSolver(kernel1, reg_lambda=1e-3, eps=1e-4, nufft_tol=1e-12)

res1 = check_solver_correctness_no_precond(solver1, x1, y1)
res1_r = check_solver_correctness_richardson(solver1, x1, y1)
res1_p = check_solver_correctness_with_precond(solver1, x1, y1, top_q=10)

rows = [
    {"metric": "rhs_relerr", "value": f"{res1['rhs_err']:.3e}"},
    {"metric": "apply_A_relerr", "value": f"{res1['A_err']:.3e}"},
    {"metric": "beta_relerr", "value": f"{res1['beta_err']:.3e}"},
    {"metric": "predict_relerr", "value": f"{res1['pred_err']:.3e}"},
    {"metric": "rich_beta_relerr", "value": f"{res1_r['beta_err']:.3e}"},
    {"metric": "rich_pred_relerr", "value": f"{res1_r['pred_err']:.3e}"},
    {"metric": "precond_beta_relerr", "value": f"{res1_p['beta_err']:.3e}"},
    {"metric": "precond_pred_relerr", "value": f"{res1_p['pred_err']:.3e}"},
]
print_metrics_table(rows, "correctness metrics (1D)")

print("=== 1D convergence metrics ===")
check_solver_convergence_metrics(solver1, x1, y1, top_q=10, pcg_tol=1e-10, pcg_maxiter=200, rich_maxiter=200)

print("=== 1D eigenspace estimation ===")
eigenspace_estimation_metrics(solver1, x1, y1, top_q=5)

print("=== 1D precond Richardson eta sweep ===")
richardson_eta_sweep(solver1, x1, y1, top_q=10, c_list=[0.2, 0.5, 0.8, 1.0, 1.2], maxiter=200)

print("=== 1D train/test RMSE ===")
x1_test = np.linspace(0.0, 1.0, 60)[:, None]
y1_test = true_func_1d(x1_test)
rmse_plain = train_test_rmse(solver1, x1, y1, x1_test, y1_test, top_q=None)
rmse_prec = train_test_rmse(solver1, x1, y1, x1_test, y1_test, top_q=10)
print("train_rmse_plain=", rmse_plain[0], "test_rmse_plain=", rmse_plain[1])
print("train_rmse_prec =", rmse_prec[0], "test_rmse_prec =", rmse_prec[1])

print("=== 2D sanity check ===")
x2, y2, _ = make_data_2d(n=40, noise=0.0, seed=1)
kernel2 = make_squared_exponential(lengthscale=0.3, dim=2, variance=1.0)
solver2 = EFGPSolver(kernel2, reg_lambda=1e-3, eps=2e-3, nufft_tol=1e-12)

res2 = check_solver_correctness_no_precond(solver2, x2, y2)
res2_r = check_solver_correctness_richardson(solver2, x2, y2)
res2_p = check_solver_correctness_with_precond(solver2, x2, y2, top_q=10)

rows = [
    {"metric": "rhs_relerr", "value": f"{res2['rhs_err']:.3e}"},
    {"metric": "apply_A_relerr", "value": f"{res2['A_err']:.3e}"},
    {"metric": "beta_relerr", "value": f"{res2['beta_err']:.3e}"},
    {"metric": "predict_relerr", "value": f"{res2['pred_err']:.3e}"},
    {"metric": "rich_beta_relerr", "value": f"{res2_r['beta_err']:.3e}"},
    {"metric": "rich_pred_relerr", "value": f"{res2_r['pred_err']:.3e}"},
    {"metric": "precond_beta_relerr", "value": f"{res2_p['beta_err']:.3e}"},
    {"metric": "precond_pred_relerr", "value": f"{res2_p['pred_err']:.3e}"},
]
print_metrics_table(rows, "correctness metrics (2D)")

print("=== 2D convergence metrics ===")
check_solver_convergence_metrics(solver2, x2, y2, top_q=10, pcg_tol=1e-10, pcg_maxiter=200, rich_maxiter=200)

print("=== 2D eigenspace estimation ===")
eigenspace_estimation_metrics(solver2, x2, y2, top_q=5)

print("=== 2D precond Richardson eta sweep ===")
richardson_eta_sweep(solver2, x2, y2, top_q=10, c_list=[0.2, 0.5, 0.8, 1.0, 1.2], maxiter=200)

print("=== 2D train/test RMSE ===")
x2_test = np.random.default_rng(9).uniform(0.0, 1.0, size=(80, 2))
y2_test = true_func_2d(x2_test)
rmse_plain = train_test_rmse(solver2, x2, y2, x2_test, y2_test, top_q=None)
rmse_prec = train_test_rmse(solver2, x2, y2, x2_test, y2_test, top_q=10)
print("train_rmse_plain=", rmse_plain[0], "test_rmse_plain=", rmse_plain[1])
print("train_rmse_prec =", rmse_prec[0], "test_rmse_prec =", rmse_prec[1])

print("=== 1D parameter sweep ===")
sweep_params_1d()

print("=== 2D parameter sweep ===")
sweep_params_2d()

check_3d_minimal()

# optional: compare with original KRR on training set
# compare_with_original_krr(solver1, x1, y1)


=== 1D sanity check ===
rhs relerr     = 7.367e-14
apply_A relerr = 4.703e-14
beta relerr    = 1.112e-09
predict relerr = 2.846e-10


richardson beta relerr    = 3.278e-01
richardson predict relerr = 3.044e-02
use top_q = 10
precond beta relerr    = 5.868e-09
precond predict relerr = 2.845e-11
correctness metrics (1D)
metric              | value    
--------------------+----------
rhs_relerr          | 7.367e-14
apply_A_relerr      | 4.703e-14
beta_relerr         | 1.112e-09
predict_relerr      | 2.846e-10
rich_beta_relerr    | 3.278e-01
rich_pred_relerr    | 3.044e-02
precond_beta_relerr | 5.868e-09
precond_pred_relerr | 2.845e-11
=== 1D convergence metrics ===
precompute_time = 0.004108499968424439
eigenspace_time = 0.0009320001117885113
precond_time    = 5.219993181526661e-05
lambda_max = 13.978719798623143
lambda_min = 0.0010000000107603347
lambda_q = 0.0013930518938088923
lambda_q_plus_1 = 0.0010578771739091298
cond(A) = 13978.71964820744
cond(PA) = 1.0578771625529502
top eigvals(A) = [13.979  7.677  5.048  2.314  0.695  0.228  0.053  0.011  0.003  0.001
  0.001  0.001  0.001  0.001  0.001]
top eigvals(PA) = [0.